# 02 — Variáveis e validação temporal

Este notebook usa as funções de `src/features.py` para criar variáveis defasadas, definir o alvo como fechamento do dia seguinte e separar treino e teste em ordem temporal.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

RAIZ = Path.cwd()
if not (RAIZ / 'src').exists() and (RAIZ.parent / 'src').exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src.features import carregar_diario, criar_features, dividir_treino_teste

diario = carregar_diario(RAIZ / 'data' / 'btc_diario.csv')
X, y = criar_features(diario)
dados = X.join(y)
dados.head()

## Conferir o alinhamento

Cada linha usa valores conhecidos até a data do índice e o alvo é o fechamento da data seguinte. Os lags evitam que informações futuras entrem nas variáveis.

In [ ]:
display(dados.tail())
print('Variáveis:', list(X.columns))
print('Alvo:', y.name)
print(f'Período após criação: {dados.index.min().date()} a {dados.index.max().date()}')
assert X.index.equals(y.index)
assert not dados.isna().any().any()
assert X.index.is_monotonic_increasing

## Divisão treino/teste

Fixe a data de corte antes de comparar modelos. O limite abaixo é um padrão inicial e deve ser confirmado pelo grupo conforme o período disponível. A data de origem no limite fica no treino; previsões emitidas depois ficam no teste.

In [ ]:
FIM_TREINO = '2024-12-31'
treino, teste = dividir_treino_teste(dados, FIM_TREINO)
X_treino, y_treino = treino[X.columns], treino[y.name]
X_teste, y_teste = teste[X.columns], teste[y.name]

print(f'Treino: {treino.index.min().date()} a {treino.index.max().date()} ({len(treino):,} linhas)')
print(f'Teste:  {teste.index.min().date()} a {teste.index.max().date()} ({len(teste):,} linhas)')
assert treino.index.max() <= pd.Timestamp(FIM_TREINO)
assert teste.index.min() > pd.Timestamp(FIM_TREINO)

In [ ]:
ax = treino[y.name].plot(figsize=(14, 5), label='Treino')
teste[y.name].plot(ax=ax, label='Teste')
ax.axvline(pd.Timestamp(FIM_TREINO), color='black', linestyle='--', label='Corte')
ax.set_title('Alvo: fechamento do dia seguinte')
ax.set_ylabel('USD')
ax.legend()
plt.show()

## Observações para a validação

A divisão preserva a ordem temporal e não embaralha as observações. Transformações que aprendam parâmetros (por exemplo, escalonamento) devem ser ajustadas somente com `X_treino` e aplicadas depois ao teste. Registre o corte e não o altere após comparar resultados.